# M2 GPU training (Kaggle)

Trains the three GPU models in parallel on **GPU T4 x2**:

| GPU | Jobs |
|---|---|
| 0 | component detector, Image2Net + CGHD (largest) |
| 1 | component detector, Image2Net only, then the crossing model (OBB) |

The orientation classifiers are trained locally, not here.

**Before running**

1. On your machine: `make data`, `make data-train`, then
   `uv run python scripts/prepare_m2_data.py` (or `--zip-only` if `data/m2/`
   already exists). This writes `data/m2_kaggle.zip`: the datasets plus a
   snapshot of `src/` with its commit.
2. Kaggle → *Datasets* → *New dataset* → upload `m2_kaggle.zip`, keep it private.
3. In this notebook: *Add input* → that dataset; *Settings* → Accelerator
   **GPU T4 x2**, Internet **on** (pretrained weights are downloaded).
4. Use **Save Version → Save & Run All** so training continues with the
   browser closed. Results appear under *Output* as `results_m2.zip`.

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

# Kaggle extracts uploaded zips; find the folder that holds code/ and datasets/.
roots = [p.parent for p in Path("/kaggle/input").rglob("CODE_COMMIT")]
assert roots, "Attach the m2_kaggle dataset (Add input) first."
SRC = roots[0].parent  # .../code/CODE_COMMIT -> dataset root
print("dataset root:", SRC)
print("code commit :", (SRC / "code" / "CODE_COMMIT").read_text().strip())

WORK = Path("/kaggle/working")
CODE = WORK / "code"
DATA = WORK / "datasets"
RESULTS = WORK / "results" / "m2"
for target, source in ((CODE, SRC / "code"), (DATA, SRC / "datasets")):
    if not target.exists():
        shutil.copytree(source, target)  # writable copies (label caches, image caches)
RESULTS.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "ultralytics==8.4.192", "scikit-learn==1.9.1"], check=True)
subprocess.run(["nvidia-smi", "-L"], check=True)

In [ ]:
ENV = dict(os.environ,
           PYTHONPATH=str(CODE / "src"),
           VCA_CODE_COMMIT=(CODE / "CODE_COMMIT").read_text().strip())

def job(task, data, name, device, epochs, batch, cache):
    return [sys.executable, "-m", "vca.training", task,
            "--data", str(DATA / data / "data.yaml"), "--name", name,
            "--device", str(device), "--epochs", str(epochs), "--batch", str(batch),
            "--workers", "2", "--cache", cache, "--output-dir", str(RESULTS)]

QUEUES = {
    0: [job("detect", "detector_i2n_cghd", "detector_i2n_cghd", 0, 80, 12, "disk")],
    1: [job("detect", "detector_i2n", "detector_i2n", 1, 100, 12, "ram"),
        job("obb", "crossing", "crossing", 1, 100, 12, "ram")],
}

# One shell chain per GPU: jobs on the same GPU run one after another,
# the two GPUs run at the same time.
procs = {}
for gpu, commands in QUEUES.items():
    log = RESULTS / f"gpu{gpu}.log"
    chain = " && ".join(" ".join(f"'{a}'" for a in cmd) for cmd in commands)
    procs[gpu] = subprocess.Popen(["bash", "-c", chain], env=ENV,
                                  stdout=open(log, "w"), stderr=subprocess.STDOUT)
    print(f"GPU {gpu}: {len(commands)} job(s), log {log}")

In [ ]:
# Wait for both GPUs, printing the latest progress line every 10 minutes.
def last_line(path):
    text = path.read_text(errors="replace").replace("\r", "\n").strip().splitlines()
    return text[-1][-160:] if text else ""

while any(p.poll() is None for p in procs.values()):
    for gpu in procs:
        code = procs[gpu].poll()
        state = "running" if code is None else f"exit {code}"
        latest = last_line(RESULTS / f"gpu{gpu}.log")
        print(f"[{time.strftime('%H:%M')}] GPU {gpu} {state}: {latest}")
    time.sleep(600)

for gpu, p in procs.items():
    print(f"GPU {gpu} finished with exit code {p.returncode}")

In [ ]:
# Held-out val metrics of every finished run, then package everything.
import json

for metrics in sorted(RESULTS.glob("*/*/metrics.json")):
    m = json.loads(metrics.read_text())
    scalars = {k: v for k, v in m.items() if not isinstance(v, (dict, list))}
    print(metrics.parent.parent.name, scalars)

# results/ holds logs, metrics, plots and weights (image caches stay in datasets/).
archive = shutil.make_archive(str(WORK / "results_m2"), "zip", WORK / "results")
print(archive, round(Path(archive).stat().st_size / 1e6), "MB")